In [1]:
import os
import time
import pandas as pd
import yfinance as yf

CACHE_DIR = "market_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SYMBOLS = {
    "EURUSD": "EURUSD=X", # M5, 15M, 1H, 4H
    "GBPUSD": "GBPUSD=X", # M5, 15M, 1H, 4H
    "USDJPY": "USDJPY=X", # M5, 15M, 1H, 4H
    "AUDUSD": "AUDUSD=X", # M5, 15M, 1H, 4H
    "USDCAD": "CAD=X", # M5, 15M, 1H, 4H
    "USDCHF": "CHF=X", # M5, 15M, 1H, 4H
    "NZDUSD": "NZDUSD=X", # M5, 15M, 1H, 4H
    "XAUUSD": "GC=F", # M5, 15M, 1H, 4H
    "BTCUSD": "BTC-USD" # M5, 15M, 1H, 4H
}

TIMEFRAMES = {
    "M5":  ("5m",  "60d"),
    "M15": ("15m", "60d"),
    "H1":  ("1h",  "730d")
}


def resample_4h(df):

    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.set_index("datetime")

    df4 = df.resample("4h").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum"
    })

    return df4.dropna().reset_index()


def download_market_data():

    for symbol, yf_symbol in SYMBOLS.items():

        print(f"\nDownloading {symbol} ({yf_symbol})")

        ticker = yf.Ticker(yf_symbol)

        for tf, (interval, period) in TIMEFRAMES.items():

            print(f"  → Fetching {tf}")

            try:

                df = ticker.history(period=period, interval=interval)

                if df.empty:
                    print("  ⚠ No data")
                    continue

                # Fix MultiIndex columns
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)

                df = df.reset_index()

                time_col = "Datetime" if "Datetime" in df.columns else "Date"

                df = df.rename(columns={
                    time_col: "datetime",
                    "Open": "open",
                    "High": "high",
                    "Low": "low",
                    "Close": "close",
                    "Volume": "volume"
                })

                df = df[["datetime","open","high","low","close","volume"]]

                df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)

                df = df.drop_duplicates("datetime")
                df = df.sort_values("datetime")

                df = df.dropna(subset=["open","high","low","close"])

                if len(df) < 10:
                    print("  ⚠ Not enough data")
                    continue

                path = f"{CACHE_DIR}/{symbol}_{tf}.parquet"
                df.to_parquet(path)

                print(f"  ✅ Saved {len(df)} bars")

                if tf == "H1":

                    df4 = resample_4h(df)

                    path = f"{CACHE_DIR}/{symbol}_H4.parquet"
                    df4.to_parquet(path)

                    print(f"  ✅ Saved {len(df4)} H4 bars")

                time.sleep(1)

            except Exception as e:

                print(f"  ❌ Error: {e}")


download_market_data()


  → Fetching M5
  ✅ Saved 16523 bars
  → Fetching M15
  ✅ Saved 5516 bars
  → Fetching H1
  ✅ Saved 17269 bars
  ✅ Saved 4446 H4 bars

  → Fetching M5
  ✅ Saved 16530 bars
  → Fetching M15
  ✅ Saved 5519 bars
  → Fetching H1
  ✅ Saved 17270 bars
  ✅ Saved 4446 H4 bars

  → Fetching M5
  ✅ Saved 16224 bars
  → Fetching M15
  ✅ Saved 5423 bars
  → Fetching H1
  ✅ Saved 17156 bars
  ✅ Saved 4440 H4 bars

  → Fetching M5
  ✅ Saved 16868 bars
  → Fetching M15
  ✅ Saved 5633 bars
  → Fetching H1
  ✅ Saved 17365 bars
  ✅ Saved 4461 H4 bars

  → Fetching M5
  ✅ Saved 16814 bars
  → Fetching M15
  ✅ Saved 5626 bars
  → Fetching H1
  ✅ Saved 17376 bars
  ✅ Saved 4461 H4 bars

  → Fetching M5
  ✅ Saved 16504 bars
  → Fetching M15
  ✅ Saved 5515 bars
  → Fetching H1
  ✅ Saved 17212 bars
  ✅ Saved 4447 H4 bars

  → Fetching M5
  ✅ Saved 16795 bars
  → Fetching M15
  ✅ Saved 5612 bars
  → Fetching H1
  ✅ Saved 17357 bars
  ✅ Saved 4460 H4 bars

  → Fetching M5
  ✅ Saved 13215 bars
  → Fetching M15
